# __MODEL_LABEL_MARKDOWN__: model exploration

Use this notebook for source exploration and disposable feature ideas.
The first section is a complete but disposable data-to-model sandbox:
load or assemble data, transform it, fit ordinary SuperGLM objects, and
inspect predictions. It deliberately sorts last and never updates the
governed model-frame artifact. Move accepted data work into
`01_data_ingestion.ipynb` and accepted model choices into
`03_model_training.ipynb`.

It also provides the temporary grouping workflow: open a published RAW
candidate in SuperGLM's editor, collapse categorical levels, then export
the actual per-feature `LevelGrouping` objects for notebook 03.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "work_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

MODEL_NAME = "__MODEL_NAME__"
MODEL_LABEL = "__MODEL_LABEL__"
DEPLOYMENT_SLOT = "__DEPLOYMENT_SLOT__"
SCRATCH_SAMPLE_ROWS = 5_000  # Set to None to use every row.
SCRATCH_RANDOM_SEED = 42
GROUPING_SOURCE_PACKAGE_VERSION = None  # None selects the latest published RAW package.
REPLACE_GROUPING_ARTIFACT = False


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

import numpy as np
import pandas as pd
from sklearn.metrics import mean_tweedie_deviance
from superglm import (
    Categorical,
    Numeric,
    Spline,
    SuperGLM,
    Tweedie,
)
from superglm.editor import EditorSession

from pricing_pipeline.modeling.scratch_benchmark import (
    fit_boosted_blend,
    superglm_edf_table,
    unconstrained_superglm_features,
)
from pricing_pipeline.notebook import (
    connect,
    export_level_groupings,
    list_candidate_versions,
    load_registered_model,
    open_candidate,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"
FRAME_ARTIFACT_PATH = MODEL_DIR / ".local" / "model_frame.joblib"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


## Sandbox 1: load or assemble disposable data

Replace this demo with any SQL query, file read, join, filter, or sample
you want to investigate. Nothing in this section is saved as the
governed model-frame handoff.


In [ ]:
rng = np.random.default_rng(SCRATCH_RANDOM_SEED)
scratch_raw = pd.DataFrame({
    "__PRIMARY_KEY__": np.arange(1, 501),
    "__FEATURE_NAME__": rng.normal(size=500),
    "segment": rng.choice(["A", "B", "C"], size=500),
})
scratch_raw["__TARGET_NAME__"] = rng.poisson(
    np.exp(
        -0.5
        + 0.25 * scratch_raw["__FEATURE_NAME__"]
        + scratch_raw["segment"].map({"A": 0.0, "B": 0.2, "C": -0.1})
    )
)
if SCRATCH_SAMPLE_ROWS is not None and len(scratch_raw) > SCRATCH_SAMPLE_ROWS:
    scratch_raw = scratch_raw.sample(
        n=SCRATCH_SAMPLE_ROWS,
        random_state=SCRATCH_RANDOM_SEED,
    )
scratch_raw = scratch_raw.sort_values("__PRIMARY_KEY__").reset_index(drop=True)
display({"Rows": len(scratch_raw), "Columns": len(scratch_raw.columns)})
display(scratch_raw.head())


In [ ]:
# Blank ingestion area: replace or extend scratch_raw however you like.
# scratch_raw = pd.read_csv("...")
# scratch_raw = pd.read_sql_query("SELECT ...", pricing.engine)


## Sandbox 2: clean and engineer disposable features

Work on `scratch_frame` so the originally loaded sample remains easy to
recover. Copy only accepted transforms into notebook 01.


In [ ]:
scratch_frame = scratch_raw.copy()
scratch_frame["candidate_transform"] = np.square(
    scratch_frame["__FEATURE_NAME__"]
)
display(scratch_frame.groupby("segment")["candidate_transform"].describe())
display(scratch_frame.head())


In [ ]:
# Blank feature area: add plots, joins, filters, or alternative columns.
# scratch_frame["another_candidate"] = ...


## Sandbox 3: define and fit a disposable model

Use normal SuperGLM feature objects here. This fits only in memory: it
does not register a model, create a manifest, build a candidate, or
publish anything. Copy accepted choices into notebook 03.


In [ ]:
SCRATCH_FAMILY = "poisson"  # For compound Tweedie: Tweedie(p=1.6)
SCRATCH_TARGET = "__TARGET_NAME__"
SCRATCH_FEATURES = {
    "__FEATURE_NAME__": Numeric(),
    "segment": Categorical(),
    "candidate_transform": Numeric(),
}
scratch_X = scratch_frame.loc[:, list(SCRATCH_FEATURES)]
scratch_y = scratch_frame[SCRATCH_TARGET].astype(float)

scratch_model = SuperGLM(
    family=SCRATCH_FAMILY,
    selection_penalty=0.0,
    features=SCRATCH_FEATURES,
).fit(scratch_X, scratch_y)


## Sandbox 4: inspect, compare, and iterate


In [ ]:
scratch_predictions = scratch_model.predict(scratch_X)
scratch_results = pd.DataFrame({
    "actual": scratch_y,
    "prediction": scratch_predictions,
})
display({
    "Rows fitted": len(scratch_results),
    "Actual mean": float(scratch_results["actual"].mean()),
    "Predicted mean": float(scratch_results["prediction"].mean()),
    "Mean absolute error": float(
        np.mean(
            np.abs(
                scratch_results["actual"]
                - scratch_results["prediction"]
            )
        )
    ),
})
display(scratch_results.head())


In [ ]:
# Blank modelling area: try another feature map, family, penalty, or split.
# alternative_features = {...}
# alternative_model = SuperGLM(...).fit(...)


## Optional: fully unconstrained SuperGLM benchmark

This is a decision-light technical benchmark: raw categorical levels,
no groupings, no monotonic or shape constraints, and data-driven spline
knots and REML lambdas. The reference level and spline penalty remain
necessary model mechanics. It stays in memory and cannot be published
or deployed from this notebook.


In [ ]:
UNCONSTRAINED_CATEGORICAL_COLUMNS = ("segment",)
UNCONSTRAINED_ORDERED_COLUMNS = {}
UNCONSTRAINED_LINEAR_COLUMNS = ()

unconstrained_features = unconstrained_superglm_features(
    scratch_X,
    categorical_columns=UNCONSTRAINED_CATEGORICAL_COLUMNS,
    ordered_columns=UNCONSTRAINED_ORDERED_COLUMNS,
    linear_columns=UNCONSTRAINED_LINEAR_COLUMNS,
    spline_kind="ps",
    k=10,
    knot_strategy="quantile_tempered",
    knot_alpha=0.2,
)
unconstrained_model = SuperGLM(
    family=SCRATCH_FAMILY,
    selection_penalty=0.0,
    features=unconstrained_features,
).fit_reml(
    scratch_X,
    scratch_y,
    runtime_validation="skip",
)
unconstrained_predictions = unconstrained_model.predict(scratch_X)
display({
    "Rows fitted": len(scratch_y),
    "Mean unit deviance": float(
        mean_tweedie_deviance(
            scratch_y,
            np.clip(unconstrained_predictions, 1e-12, None),
            power=float(getattr(unconstrained_model.family, "p", 1.0)),
        )
    ),
    "Evidence": "in-sample technical fit; use OOF evidence below for comparison",
})
display(superglm_edf_table(unconstrained_model))


## Optional: CatBoost + LightGBM + XGBoost blend

Install once with `uv sync --extra scratch`, restart the kernel, then
run this cell. Each learner is fitted out-of-fold and non-negative
convex blend weights are learned only from those held-out predictions.
These weights are a technical predictive benchmark, not a proposed
pricing blend. A governed GAM/GBM blend should impose its chosen GAM
floor and be assessed through tail, calibration, and stability evidence.
This remains scratch-only: it does not touch SQL or the candidate lane.


In [ ]:
boosted_blend = fit_boosted_blend(
    scratch_X,
    scratch_y,
    categorical_columns=("segment",),
    n_splits=5,
    random_state=SCRATCH_RANDOM_SEED,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    reference_superglm=unconstrained_model,
)
display(boosted_blend.metrics.sort_values("mean_unit_deviance"))
display(pd.Series(boosted_blend.weights, name="OOF convex weight"))
display(boosted_blend.oof_predictions.head())


## Optional: create the routine level-grouping artifact

This section is read-only in SQL but requires `DATABASE_MODE = "remote"`
because editable candidate bundles are governed by the remote workbench.
Select levels in the widget and use **Collapse and refit** as often as
needed, across as many categorical features as needed.


In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_candidate_versions(pricing, model=model)
raw_versions = versions.loc[versions["Kind"].eq("RAW")].copy()
display(raw_versions)
if raw_versions.empty:
    raise LookupError("No published RAW candidate is available for grouping.")
selected_package_version = (
    int(raw_versions.iloc[0]["Package"])
    if GROUPING_SOURCE_PACKAGE_VERSION is None
    else int(GROUPING_SOURCE_PACKAGE_VERSION)
)
if selected_package_version not in set(
    raw_versions["Package"].astype(int)
):
    raise ValueError(
        "GROUPING_SOURCE_PACKAGE_VERSION is not in the displayed RAW list."
    )
grouping_candidate = open_candidate(
    pricing,
    model=model,
    package_version=selected_package_version,
)


In [ ]:
grouping_session = EditorSession.from_model(
    grouping_candidate.bundle.fitted_model,
    train_data=(
        grouping_candidate.bundle.X,
        grouping_candidate.bundle.y,
        grouping_candidate.bundle.sample_weight,
        grouping_candidate.bundle.offset,
    ),
    cv_report=grouping_candidate.bundle.cv_report,
)
display(grouping_session.widget())


## Export all current groupings

Run this only after every intended collapse/refit has completed. The
binary contains actual `LevelGrouping` objects; its JSON sidecar is
generated integrity and lineage evidence, not an analyst config file.


In [ ]:
grouping_artifact = export_level_groupings(
    grouping_candidate,
    editor_session=grouping_session,
    path=GROUPING_ARTIFACT_PATH,
    replace=REPLACE_GROUPING_ARTIFACT,
)
display(grouping_artifact)
